# Hamiltonian Flow Matching — Entropy Potential (MNIST)

Trains a UNet to transport Gaussian noise → MNIST digits under the **entropy potential**
`F(ρ) = β ∫ ρ log ρ dx`.

The variance schedule σ(t) is pre-computed once by `DensityGaussianPath` at construction
(200 RK4 steps). During training, σ(t) and σ'(t) are evaluated from the cached schedule
via `path.schedule.evaluate(t)` and broadcast over the 4D image tensors.

**Workflow:**
1. Precompute σ(t) schedule — one call at construction time.
2. Train UNet on OT-coupled (noise, MNIST) pairs using the schedule.
3. Evaluate with `NeuralODE` and measure pixel-space Fréchet distance.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../../'))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchdyn.core import NeuralODE
from torchvision import datasets, transforms
from torchvision.transforms import ToPILImage
from torchvision.utils import make_grid

from torchcfm.hamiltonian import (
    EntropyPotential, DensityGaussianPath, HamiltonianFlowMatcher,
    flow_matching_loss, frechet_distance,
)
from torchcfm.models.unet import UNetModel
from torchcfm.optimal_transport import OTPlanSampler

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

batch_size = 128
n_epochs   = 10

# Entropy potential hyperparameters (matched to original notebook)
beta    = 0.01
coeff   = 1.5
sigma_0 = 0.15

savedir = 'models/mnist_entropy'
os.makedirs(savedir, exist_ok=True)

print(f'device: {device}')
print(f'EntropyPotential: beta={beta}, coeff={coeff}, sigma_0={sigma_0}')

## MNIST dataset

In [ ]:
trainset = datasets.MNIST(
    '../data', train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ]),
)
train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, drop_last=True
)
print(f'Training samples: {len(trainset)}, batches per epoch: {len(train_loader)}')

## Entropy potential and σ(t) schedule

`DensityGaussianPath` precomputes the σ(t) schedule by integrating the entropy ODE
`σ' = -√(2β log(σ) + C)` once with 200 RK4 steps. The schedule is then evaluated via
linear interpolation during training.

In [ ]:
density_pot = EntropyPotential(beta=beta, coeff=coeff)
path        = DensityGaussianPath(density_pot, sigma_0=sigma_0, n_steps=200, method='rk4')
ot_sampler  = OTPlanSampler(method='exact')

sched = path.schedule
print(f'sigma schedule: t=[{sched.t[0]:.2f}, {sched.t[-1]:.2f}],  '
      f'sigma=[{sched.sigma[0].item():.4f}, {sched.sigma[-1].item():.4f}]')

In [ ]:
# Visualise the precomputed σ(t) schedule
t_np     = sched.t.numpy()
sigma_np = sched.sigma.squeeze(-1).numpy()

plt.figure(figsize=(6, 3))
plt.plot(t_np, sigma_np)
plt.xlabel('t'); plt.ylabel('σ(t)')
plt.title(f'Entropy σ(t) schedule  (beta={beta}, coeff={coeff})')
plt.tight_layout(); plt.show()

## UNet model

In [ ]:
model     = UNetModel(dim=(1, 28, 28), num_channels=32, num_res_blocks=1).to(device)
optimizer = torch.optim.Adam(model.parameters())

n_params = sum(p.numel() for p in model.parameters())
print(f'UNet parameters: {n_params:,}')

## Training

For each mini-batch:
1. OT-couple `(x0 ~ N(0,I), x1 ~ MNIST)`
2. Sample `t ~ Uniform(0,1)` and evaluate `σ(t), σ'(t)` from the precomputed schedule
3. Compute `x_t = μ_t + σ(t) ε` and the conditional velocity `u_t`
4. Minimise `‖v_θ(t, x_t) - u_t‖²`

In [ ]:
model.train()
losses = []

for epoch in range(n_epochs):
    epoch_loss = 0.0
    for x1, _ in train_loader:
        optimizer.zero_grad()

        x1 = x1.to(device)           # (B, 1, 28, 28)
        x0 = torch.randn_like(x1)    # Gaussian noise source

        # OT coupling
        x0, x1 = ot_sampler.sample_plan(x0, x1)

        # Sample t and look up σ(t), σ'(t) from the precomputed schedule
        t_flat = torch.rand(x1.shape[0], device=device)          # (B,)
        sigma_t, sigma_prime_t = path.schedule.evaluate(t_flat)  # (B,) each

        # Broadcast over image dimensions
        t_r         = t_flat.view(-1, 1, 1, 1)
        sigma_r     = sigma_t.view(-1, 1, 1, 1)
        sigma_r_p   = sigma_prime_t.view(-1, 1, 1, 1)

        epsilon = torch.randn_like(x0)
        mu_t    = (1 - t_r) * x0 + t_r * x1
        mu_t_p  = x1 - x0

        xt = mu_t + sigma_r * epsilon
        ut = sigma_r_p * (xt - mu_t) / (sigma_r + 1e-8) + mu_t_p

        # UNet receives (t_1d, x) — time as 1D tensor
        vt   = model(t_flat, xt)
        loss = flow_matching_loss(vt, ut)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        losses.append(loss.item())

    print(f'epoch {epoch + 1:2d}/{n_epochs}: avg loss = {epoch_loss / len(train_loader):.5f}')

In [ ]:
plt.figure(figsize=(8, 3))
plt.semilogy(losses)
plt.xlabel('iteration'); plt.ylabel('loss'); plt.title('Training loss')
plt.tight_layout(); plt.show()

In [ ]:
torch.save(model.state_dict(), os.path.join(savedir, 'model.pt'))
print(f'Model saved to {savedir}/model.pt')

## Evaluation — generated samples

In [ ]:
model.eval()
node = NeuralODE(model, solver='dopri5', sensitivity='adjoint', atol=1e-4, rtol=1e-4)

with torch.no_grad():
    traj = node.trajectory(
        torch.randn(100, 1, 28, 28, device=device),
        t_span=torch.linspace(0, 1, 100, device=device),
    )

grid = make_grid(
    traj[-1, :100].view(-1, 1, 28, 28).clip(-1, 1),
    value_range=(-1, 1), padding=0, nrow=10,
)
img = ToPILImage()(grid)
plt.figure(figsize=(10, 10))
plt.imshow(img); plt.axis('off')
plt.title('Generated MNIST digits (100 samples)')
plt.tight_layout(); plt.show()

In [ ]:
# Generation process: image grids at five time steps (noise → digits)
t_steps = [0, 24, 49, 74, 99]
step_labels = ['t = 0.00  (noise)', 't = 0.25', 't = 0.50', 't = 0.75', 't = 1.00  (generated)']

fig, axes = plt.subplots(1, len(t_steps), figsize=(20, 4))
for ax, ti, lbl in zip(axes, t_steps, step_labels):
    grid_t = make_grid(
        traj[ti, :25].view(-1, 1, 28, 28).clip(-1, 1),
        value_range=(-1, 1), padding=1, nrow=5,
    )
    ax.imshow(ToPILImage()(grid_t))
    ax.set_title(lbl, fontsize=11); ax.axis('off')
plt.suptitle('Entropy flow: denoising process  (25 samples)', fontsize=14)
plt.tight_layout(); plt.show()

## FID evaluation (pixel-space)

We use `frechet_distance` from the library which computes the Fréchet distance
between two Gaussian summaries.  Here we work in pixel space (784D) rather than
Inception feature space — it provides a useful relative comparison across runs.

In [ ]:
n_fid = batch_size

# Real images — one batch from loader
real_imgs, _ = next(iter(train_loader))
real_flat = real_imgs.view(n_fid, -1).numpy()  # (N, 784)

# Generated images — ODE from noise
with torch.no_grad():
    traj_fid = node.trajectory(
        torch.randn(n_fid, 1, 28, 28, device=device),
        t_span=torch.linspace(0, 1, 100, device=device),
    )
gen_flat = traj_fid[-1].cpu().view(n_fid, -1).numpy()  # (N, 784)

# Gaussian summaries in pixel space
mu_real  = real_flat.mean(axis=0)
cov_real = np.cov(real_flat, rowvar=False)
mu_gen   = gen_flat.mean(axis=0)
cov_gen  = np.cov(gen_flat, rowvar=False)

fid = frechet_distance(mu_real, cov_real, mu_gen, cov_gen)
print(f'Pixel-space Fréchet distance: {fid:.4f}')